In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "run_simulation.py").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))


# Sensitivity analysis

Sweeps **review-duration multipliers** and **pre-application distribution** choices to see which assumptions move outcomes most.

**Outputs:** per-run and summary CSVs under `results/`.

Start with small `N_RUNS` and a reduced parameter grid while testing.


## 1. Setup

Defines the sensitivity runner function.


In [ ]:
import itertools
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from run_simulation import run_simulation
from visualize_permits import calculate_step_waiting_service_totals

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

### Sensitivity runner


In [ ]:
def run_duration_level_sensitivity(
    n_runs: int,
    num_permits: int,
    levels: list[float],
    pre_app_distributions: list[str],
    sequential: str = "standard",
    base_seed: int = 42,
    planning_staff_baseline: int = 20,
    public_works_staff_baseline: int = 30,
    fire_staff_baseline: int = 10,
    planning_caseload_per_staff: float = 7,
    public_works_caseload_per_staff: float = 7,
    fire_caseload_per_staff: float = 7,
):
    rows = []

    stages = [
        "planning",
        "public_works",
        "fire",
        "special_zoning",
        "agency_referral",
    ]
    step_names = [
        "EPA Debris",
        "USACE Debris",
        "Pre-Application Activities",
        "Applicant Revisions",
        "Planning",
        "Agency Referral",
        "Special Zoning",
        "Public Works",
        "Fire Review",
    ]
    level_combos = list(itertools.product(levels, repeat=len(stages)))

    total_runs = len(pre_app_distributions) * len(level_combos) * n_runs
    completed_runs = 0
    start_time = time.time()

    for pre_app_dist in pre_app_distributions:
        for combo_vals in level_combos:
            multiplier_map = dict(zip(stages, combo_vals))
            planning_staff_count = max(1, int(round(planning_staff_baseline)))
            public_works_staff_count = max(1, int(round(public_works_staff_baseline)))
            fire_staff_count = max(1, int(round(fire_staff_baseline)))

            combo_name = (
                f"pre_app={pre_app_dist} | "
                + " | ".join(f"{k}={v:.1f}x" for k, v in multiplier_map.items())
            )
            combo_name += (
                f" | planning_staff={planning_staff_count}"
                f" | public_works_staff={public_works_staff_count}"
                f" | fire_staff={fire_staff_count}"
            )

            for run_idx in range(n_runs):
                seed = base_seed + run_idx
                sim = run_simulation(
                    num_permits=num_permits,
                    random_seed=seed,
                    sequential=sequential,
                    review_duration_multipliers=multiplier_map,
                    pre_application_distribution=pre_app_dist,
                    planning_staff_count=planning_staff_count,
                    public_works_staff_count=public_works_staff_count,
                    planning_caseload_per_staff=planning_caseload_per_staff,
                    public_works_caseload_per_staff=public_works_caseload_per_staff,
                    fire_staff_count=fire_staff_count,
                    fire_caseload_per_staff=fire_caseload_per_staff,
                )
                stats = sim.get_statistics()
                completed_runs += 1
                if completed_runs % 1000 == 0 or completed_runs == total_runs:
                    elapsed = time.time() - start_time
                    avg_per_run = elapsed / completed_runs if completed_runs else 0.0
                    remaining = max(total_runs - completed_runs, 0)
                    eta_sec = remaining * avg_per_run
                    print(
                        f"Progress: {completed_runs}/{total_runs} runs "
                        f"({100.0 * completed_runs / total_runs:.1f}%) | "
                        f"elapsed {elapsed / 60:.1f} min | ETA {eta_sec / 60:.1f} min"
                    )

                if "message" in stats:
                    continue

                avg_total = stats.get("average_total_time") or {}
                total_wait = stats.get("total_waiting_time") or {}
                total_service = stats.get("total_service_time") or {}
                county_vs_app = stats.get("county_review_vs_applicant") or {}

                # Mean waiting/service time per permit for each step
                permits = sim.completed_permits
                n_permits = len(permits)

                # Planning-to-ready metric: from planning start to ready for construction
                planning_to_ready_days = [
                    (p.ready_for_construction - p.planning_request)
                    for p in permits
                    if p.planning_request is not None and p.ready_for_construction is not None
                ]
                mean_planning_to_ready_days = (
                    float(np.mean(planning_to_ready_days))
                    if planning_to_ready_days
                    else np.nan
                )

                # Debris gating metric: if debris ends after ready_for_construction,
                # permit would wait this additional time before construction can truly start.
                debris_blocking_wait_days = [
                    (p.debris_removal_end - p.ready_for_construction)
                    for p in permits
                    if p.debris_removal_end is not None
                    and p.ready_for_construction is not None
                    and p.debris_removal_end > p.ready_for_construction
                ]
                debris_blocking_wait_count = len(debris_blocking_wait_days)
                debris_blocking_wait_mean = (
                    float(np.mean(debris_blocking_wait_days))
                    if debris_blocking_wait_days
                    else 0.0
                )
                debris_blocking_wait_median = (
                    float(np.median(debris_blocking_wait_days))
                    if debris_blocking_wait_days
                    else 0.0
                )
                debris_blocking_wait_max = (
                    float(np.max(debris_blocking_wait_days))
                    if debris_blocking_wait_days
                    else 0.0
                )

                wait_sums = {s: 0.0 for s in step_names}
                service_sums = {s: 0.0 for s in step_names}
                for p in permits:
                    step_data = calculate_step_waiting_service_totals(p)
                    for s in step_names:
                        vals = step_data.get(s, {"waiting": 0.0, "service": 0.0})
                        wait_sums[s] += vals.get("waiting", 0.0)
                        service_sums[s] += vals.get("service", 0.0)

                step_cols = {}
                for s in step_names:
                    col_key = s.lower().replace(" ", "_").replace("-", "_")
                    denom = n_permits if n_permits > 0 else 1
                    step_cols[f"step_wait_mean__{col_key}"] = wait_sums[s] / denom
                    step_cols[f"step_service_mean__{col_key}"] = service_sums[s] / denom

                rows.append(
                    {
                        "combo": combo_name,
                        "pre_application_distribution": pre_app_dist,
                        "planning_staff_count": planning_staff_count,
                        "public_works_staff_count": public_works_staff_count,
                        "fire_staff_count": fire_staff_count,
                        "planning_level": multiplier_map["planning"],
                        "public_works_level": multiplier_map["public_works"],
                        "fire_level": multiplier_map["fire"],
                        "special_zoning_level": multiplier_map["special_zoning"],
                        "agency_referral_level": multiplier_map["agency_referral"],
                        "run": run_idx,
                        "seed": seed,
                        "completed_permits": n_permits,
                        "mean_total_days": avg_total.get("mean", np.nan),
                        "mean_application_to_ready_days": mean_planning_to_ready_days,
                        "planning_to_ready_n": len(planning_to_ready_days),
                        "debris_blocking_wait_count": debris_blocking_wait_count,
                        "debris_blocking_wait_share": (
                            debris_blocking_wait_count / n_permits if n_permits > 0 else np.nan
                        ),
                        "debris_blocking_wait_mean_days": debris_blocking_wait_mean,
                        "debris_blocking_wait_median_days": debris_blocking_wait_median,
                        "debris_blocking_wait_max_days": debris_blocking_wait_max,
                        "median_total_days": avg_total.get("median", np.nan),
                        "std_total_days": avg_total.get("std", np.nan),
                        "min_total_days": avg_total.get("min", np.nan),
                        "max_total_days": avg_total.get("max", np.nan),
                        "mean_wait_days": total_wait.get("mean", np.nan),
                        "mean_service_days": total_service.get("mean", np.nan),
                        "county_review_mean": county_vs_app.get("county_review_mean", np.nan),
                        "applicant_mean": county_vs_app.get("applicant_mean", np.nan),
                        "debris_mean": county_vs_app.get("debris_mean", np.nan),
                        "stats_json": json.dumps(stats),
                        **step_cols,
                    }
                )

    return pd.DataFrame(rows)

## 2. Run sensitivity sweep


In [ ]:
# --- Experiment settings ---
levels = [0.5, 1.0, 2.0]  # 50%, 100%, 150%
pre_app_distributions = ["lognormal_180", "lognormal_60", "lognormal_10"]
# Add "baseline" to the list above if you want to include the original model behavior.

N_RUNS = 10
N_PERMITS = 1200
SEQUENTIAL_MODE = "standard"  # "standard", "parallel", or "sequential"

# Staff counts stay fixed at these baseline values across all scenarios.
PLANNING_STAFF_BASELINE = 8
PUBLIC_WORKS_STAFF_BASELINE = 12
FIRE_STAFF_BASELINE = 4
PLANNING_CASELOAD_PER_STAFF = 7
PUBLIC_WORKS_CASELOAD_PER_STAFF = 7
FIRE_CASELOAD_PER_STAFF = 7

results_df = run_duration_level_sensitivity(
    n_runs=N_RUNS,
    num_permits=N_PERMITS,
    levels=levels,
    pre_app_distributions=pre_app_distributions,
    sequential=SEQUENTIAL_MODE,
    base_seed=42,
    planning_staff_baseline=PLANNING_STAFF_BASELINE,
    public_works_staff_baseline=PUBLIC_WORKS_STAFF_BASELINE,
    fire_staff_baseline=FIRE_STAFF_BASELINE,
    planning_caseload_per_staff=PLANNING_CASELOAD_PER_STAFF,
    public_works_caseload_per_staff=PUBLIC_WORKS_CASELOAD_PER_STAFF,
    fire_caseload_per_staff=FIRE_CASELOAD_PER_STAFF,
)

print(f"Rows: {len(results_df)}")
print(f"Unique combinations: {results_df['combo'].nunique()}")
print(f"Expected combinations: {len(pre_app_distributions) * (len(levels) ** 5)}")
results_df.head()

## 3. Review summary table


In [ ]:
summary = (
    results_df
    .groupby([
        "pre_application_distribution",
        "planning_level",
        "public_works_level",
        "fire_level",
        "special_zoning_level",
        "agency_referral_level",
    ], as_index=False)
    .mean(numeric_only=True)
    .sort_values("mean_total_days")
    .reset_index(drop=True)
)

summary = summary.rename(columns={"run": "n_runs_proxy"})

print("Top 10 fastest combinations (by mean total days):")
display(summary.head(10))

print("Top 10 slowest combinations:")
display(summary.tail(10).sort_values("mean_total_days", ascending=False))

## 4. Optional distribution overlay plot


In [ ]:
# Optional visualization: compare pre-application distributions at baseline review levels
from simulation_plot_helpers import show_boxplot_stats_table

plot_df = summary[
    (summary["planning_level"] == 1.0)
    & (summary["public_works_level"] == 1.0)
    & (summary["fire_level"] == 1.0)
    & (summary["special_zoning_level"] == 1.0)
    & (summary["agency_referral_level"] == 1.0)
].copy()

base_mask = (
    (results_df["planning_level"] == 1.0)
    & (results_df["public_works_level"] == 1.0)
    & (results_df["fire_level"] == 1.0)
    & (results_df["special_zoning_level"] == 1.0)
    & (results_df["agency_referral_level"] == 1.0)
)
run_df = results_df.loc[base_mask].copy()

pre_keys = plot_df.sort_values("mean_total_days")["pre_application_distribution"].tolist()
data = [
    run_df.loc[run_df["pre_application_distribution"] == p, "mean_total_days"].dropna().to_numpy()
    for p in pre_keys
]

fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot(
    data,
    labels=pre_keys,
    patch_artist=True,
    showfliers=True,
    whis=1.5,
)
for patch in bp["boxes"]:
    patch.set_facecolor("#81C784")
    patch.set_edgecolor("black")
    patch.set_alpha(0.9)
ax.set_title(
    "Total days (disaster → construction): within-run mean; box across runs "
    "(review multipliers at baseline)"
)
ax.set_xlabel("Pre-application distribution")
ax.set_ylabel("Mean total days per run")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

show_boxplot_stats_table(
    [
        (p, run_df.loc[run_df["pre_application_distribution"] == p, "mean_total_days"].dropna().to_numpy())
        for p in pre_keys
    ],
    heading="Mean total days per run (disaster → construction) by pre-application distribution",
)


## 5. Export results


In [ ]:
# Export CSV outputs (every run + grouped summary) into Results folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
from repo_paths import RESULTS_DIR

results_dir = RESULTS_DIR
results_dir.mkdir(parents=True, exist_ok=True)

runs_csv = results_dir / f"sensitivity_runs_all_combinations_{timestamp}.csv"
summary_csv = results_dir / f"sensitivity_summary_all_combinations_{timestamp}.csv"

results_df.to_csv(runs_csv, index=False)
summary.to_csv(summary_csv, index=False)

print(f"Saved per-run results: {runs_csv}")
print(f"Saved grouped summary: {summary_csv}")
print(f"Per-run rows exported: {len(results_df)}")
print(f"Unique parameter combinations: {summary.shape[0]}")

summary.head()
